<a href="https://colab.research.google.com/github/Kantheephob/AI-Projects/blob/main/Steam_Association_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Dataset

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "steam-200k.csv" # ชื่อไฟล์ที่จะโหลดมาจาก api

df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "tamber/steam-video-games", file_path)
df.columns = ['user_id', 'game_name', 'action', 'play_time', '-']

df.head()

Using Colab cache for faster access to the 'steam-video-games' dataset.


,user_id,game_name,action,play_time,-
0,151603712,The Elder Scrolls V Skyrim,play,273.0,0
1,151603712,Fallout 4,purchase,1.0,0
2,151603712,Fallout 4,play,87.0,0
3,151603712,Spore,purchase,1.0,0
4,151603712,Spore,play,14.9,0


# Data Preprocessing

In [ ]:
df.drop(['play_time', '-'], axis=1, inplace=True)
df.head()

,user_id,game_name,action
0,151603712,The Elder Scrolls V Skyrim,play
1,151603712,Fallout 4,purchase
2,151603712,Fallout 4,play
3,151603712,Spore,purchase
4,151603712,Spore,play


In [ ]:
# Group by 'user_id' and aggregate 'game_name' into a list for each user
games_per_user = df.groupby('user_id')['game_name'].apply(list).reset_index()

# Remove duplicates within each list of games
games_per_user['game_name'] = games_per_user['game_name'].apply(lambda x: list(set(x)))

# Display the result
display(games_per_user.head())

,user_id,game_name
0,5250,"[Cities Skylines, Portal 2, Half-Life 2 Episod..."
1,76767,"[Call of Duty Black Ops - Multiplayer, Thief D..."
2,86540,"[Skyrim High Resolution Texture Pack, Sam & Ma..."
3,103360,"[Ricochet, Half-Life Opposing Force, Half-Life..."
4,144736,"[Half-Life Opposing Force, Half-Life, Team For..."


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

game_matrix = mlb.fit_transform(games_per_user['game_name'])

input_df = pd.DataFrame(game_matrix, columns=mlb.classes_).astype(bool)

display(input_df.head())
print(input_df.shape)

,007 Legends,0RBITALIS,1... 2... 3... KICK IT! (Drop That Beat Like an Ugly Baby),10 Second Ninja,"10,000,000",100% Orange Juice,1000 Amps,12 Labours of Hercules,12 Labours of Hercules II The Cretan Bull,12 Labours of Hercules III Girl Power,...,rFactor 2,realMyst,realMyst Masterpiece Edition,resident evil 4 / biohazard 4,rymdkapsel,sZone-Online,samurai_jazz,the static speaks my name,theHunter,theHunter Primal
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


(12393, 5155)


# Association Mining

In [ ]:
pip install mlxtend

In [ ]:
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
import time
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

supports = np.arange(0.1, 0.009, -0.01).tolist()
confidences = np.arange(0.5, 0.09, -0.1).tolist()

apriori_times = []
fpgrowth_times = []
rules_list = []

for s in supports:
    # --- Step 1: รันแยกกันเพื่อจับเวลา ---
    # FP-Growth
    start_fpg = time.time()
    freq_fpg = fpgrowth(input_df, min_support=s, use_colnames=True)
    time_fpg = time.time() - start_fpg

    # Apriori
    start_apr = time.time()
    freq_apr = apriori(input_df, min_support=s, use_colnames=True)
    time_apr = time.time() - start_apr

    # --- Step 2: สร้างกฎแยกตามอัลกอริทึม ---
    for c in confidences:
        # เก็บผลของ FP-Growth
        if not freq_fpg.empty:
            rules_fpg = association_rules(freq_fpg, metric="confidence", min_threshold=c)
            rules_list.append({
                'support': s, 'confidence': c, 'algorithm': 'fpgrowth',
                'time': time_fpg, 'num_rules': len(rules_fpg),
                'max_lift': rules_fpg['lift'].max() if not rules_fpg.empty else 0
            })

        # เก็บผลของ Apriori
        if not freq_apr.empty:
            rules_apr = association_rules(freq_apr, metric="confidence", min_threshold=c)
            rules_list.append({
                'support': s, 'confidence': c, 'algorithm': 'apriori',
                'time': time_apr, 'num_rules': len(rules_apr),
                'max_lift': rules_apr['lift'].max() if not rules_apr.empty else 0
            })

    print(f"Done Support {s:.2f} | FPG Time: {time_fpg:.4f}s | Apr Time: {time_apr:.4f}s")

Done Support 0.10 | FPG Time: 3.0816s | Apr Time: 0.1147s
Done Support 0.09 | FPG Time: 4.7152s | Apr Time: 0.1204s
Done Support 0.08 | FPG Time: 5.6905s | Apr Time: 0.1679s
Done Support 0.07 | FPG Time: 4.8275s | Apr Time: 0.1188s
Done Support 0.06 | FPG Time: 3.8181s | Apr Time: 0.1127s
Done Support 0.05 | FPG Time: 3.0984s | Apr Time: 0.1343s
Done Support 0.04 | FPG Time: 3.0380s | Apr Time: 0.2055s
Done Support 0.03 | FPG Time: 4.0192s | Apr Time: 0.3955s
Done Support 0.02 | FPG Time: 23.9836s | Apr Time: 3.0617s
Done Support 0.01 | FPG Time: 424.8934s | Apr Time: 75.0390s


In [ ]:
results_df = pd.DataFrame(rules_list)
results_df

,support,confidence,algorithm,time,num_rules,max_lift
0,0.10,0.5,fpgrowth,3.081633,0,0.000000
1,0.10,0.5,apriori,0.114738,0,0.000000
2,0.10,0.4,fpgrowth,3.081633,0,0.000000
3,0.10,0.4,apriori,0.114738,0,0.000000
4,0.10,0.3,fpgrowth,3.081633,0,0.000000
...,...,...,...,...,...,...
95,0.01,0.3,apriori,75.039015,604722,96.820312
96,0.01,0.2,fpgrowth,424.893400,628156,96.820312
97,0.01,0.2,apriori,75.039015,628156,96.820312
98,0.01,0.1,fpgrowth,424.893400,642742,96.820312


## 4.1

In [ ]:
best_configs = results_df.sort_values(by='max_lift', ascending=False)
display(best_configs.head(10))

,support,confidence,algorithm,time,num_rules,max_lift
96,0.01,0.2,fpgrowth,424.893400,628156,96.820312
97,0.01,0.2,apriori,75.039015,628156,96.820312
99,0.01,0.1,apriori,75.039015,642742,96.820312
98,0.01,0.1,fpgrowth,424.893400,642742,96.820312
91,0.01,0.5,apriori,75.039015,540339,96.820312
90,0.01,0.5,fpgrowth,424.893400,540339,96.820312
94,0.01,0.3,fpgrowth,424.893400,604722,96.820312
95,0.01,0.3,apriori,75.039015,604722,96.820312
92,0.01,0.4,fpgrowth,424.893400,578669,96.820312
93,0.01,0.4,apriori,75.039015,578669,96.820312


กำหนดค่า minimum support เป็น 0.01 และ minimum confidence เป็น 0.2 อ้างอิงจากการทำ gridsearch ในช่วง support = 0.1->0.01 และ confidence = 0.5->0.1โดยดูจากค่า max_lift เป็นที่ตั้ง

In [ ]:
final_freq_list = apriori(input_df, min_support=0.01, use_colnames=True)
final_rules = association_rules(final_freq_list, metric="confidence", min_threshold=0.2)

## 4.2

In [ ]:
high_support_low_lift_rules = final_rules[(final_rules['lift'] < 1.0)]
display(high_support_low_lift_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('support', ascending=False).head(20))

,antecedents,consequents,support,confidence,lift
500,(Team Fortress 2),(Dota 2),0.058339,0.311235,0.796765
510,(Unturned),(Dota 2),0.045267,0.358925,0.918851
357,(Counter-Strike Source),(Dota 2),0.023400,0.296524,0.759103
502,(The Elder Scrolls V Skyrim),(Dota 2),0.021464,0.370990,0.949738
191,(Counter-Strike),(Dota 2),0.019689,0.285047,0.729722
477,(Half-Life 2 Lost Coast),(Dota 2),0.017752,0.224261,0.574110
1481,"(Counter-Strike, Counter-Strike Condition Zero)",(Dota 2),0.017268,0.315169,0.806836
256,(Counter-Strike Condition Zero Deleted Scenes),(Dota 2),0.017268,0.315169,0.806836
228,(Counter-Strike Condition Zero),(Dota 2),0.017268,0.315169,0.806836
1567,"(Counter-Strike, Counter-Strike Condition Zero...",(Dota 2),0.017268,0.315169,0.806836


มี Spurious Correlation ดูได้จาก rule ที่มีค่า support สูงๆ แต่ lift <= 1 จะเห็นได้ว่าไม่ว่าจะซื้อหรือเล่นเกมอะไรก็จะมี Dota2 พ่วงมาด้วย ซึ่งส่วนใหญ่จะดูไม่ค่อยสัมพันธ์กับเกมด้านหน้าในด้านแนวเกม แต่เพราะ Dota2 เป็นเกมที่ได้รับความนิยมและเป็นเกมฟรีเลยมีโอกาศสูงที่จะไปอยู่ในหลาย translation

## 4.3

In [ ]:
gold_rules = final_rules.sort_values('lift', ascending=False).head(20)
display(gold_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

,antecedents,consequents,support,confidence,lift
73734,"(Half-Life Blue Shift, Half-Life, Counter-Stri...","(Half-Life Opposing Force, Counter-Strike Cond...",0.010328,1.000000,96.820312
73741,"(Counter-Strike, Half-Life Opposing Force, Cou...","(Half-Life Blue Shift, Half-Life, Counter-Stri...",0.010328,1.000000,96.820312
156837,"(Half-Life, Team Fortress Classic, Counter-Str...","(Half-Life Opposing Force, Counter-Strike Cond...",0.010248,1.000000,96.820312
156841,"(Half-Life, Team Fortress Classic, Half-Life B...","(Half-Life Opposing Force, Counter-Strike Cond...",0.010248,1.000000,96.820312
156852,"(Half-Life Opposing Force, Half-Life, Counter-...","(Team Fortress Classic, Half-Life Blue Shift, ...",0.010248,0.992188,96.820312
73745,"(Half-Life Opposing Force, Counter-Strike, Cou...","(Half-Life Blue Shift, Half-Life, Counter-Stri...",0.010328,1.000000,96.820312
74033,"(Half-Life Blue Shift, Team Fortress Classic, ...","(Counter-Strike, Half-Life Opposing Force, Cou...",0.010248,1.000000,96.820312
74019,"(Counter-Strike, Half-Life Opposing Force, Cou...","(Half-Life Blue Shift, Team Fortress Classic, ...",0.010248,0.992188,96.820312
74012,"(Counter-Strike, Team Fortress Classic, Half-L...","(Half-Life Opposing Force, Counter-Strike Cond...",0.010248,1.000000,96.820312
74026,"(Half-Life Opposing Force, Counter-Strike, Cou...","(Team Fortress Classic, Half-Life Blue Shift, ...",0.010248,0.992188,96.820312


มีการค้นพบ Surprising Pattern/Knowledge จากค่า rule ที่มีค่า lift สูงมาก (มากกว่า 1 อย่างชัดเจน) และมีค่า confidence สูง ซึ่งบ่งบอกถึงความสัมพันธ์ที่แข็งแรงระหว่างรายการสินค้า จากผลลัพธ์พบว่า ผู้ที่เคยซื้อหรือเล่นเกมจากค่าย Valve มาก่อน มีแนวโน้มสูงที่จะซื้อหรือเล่นเกมอื่น ๆ จากค่ายเดียวกัน ซึ่งสะท้อนถึงพฤติกรรมของผู้บริโภคที่นิยมเกมภายใน ecosystem เดียวกัน

## 4.4

In [ ]:
print(f'{gold_rules.iloc[0]['antecedents']} -> {gold_rules.iloc[0]["consequents"]}')

frozenset({'Half-Life Blue Shift', 'Half-Life', 'Counter-Strike', 'Counter-Strike Condition Zero Deleted Scenes'}) -> frozenset({'Half-Life Opposing Force', 'Counter-Strike Condition Zero'})


อธิบายได้ว่าเกมทั้งหมดนี้เป็นเกมจากค่าย Valve ที่มี ecosystem เดียวกัน คือใช้เกม engine เดียวกันในการสร้างทำให้ผู้เล่นมีความคุ้นเคยและไม่ต้องปรับตัวมากเวลาเล่นเกมใหม่ในค่ายเดียวกัน ผู้เล่นที่เคยเล่นหรือซื้อเกมในกลุ่มนี้มาก่อนมักมีความสนใจในเนื้อเรื่องและ gameplay ที่ต่อเนื่อง ทำให้มีแนวโน้มสูงที่จะซื้อหรือเล่นภาคเสริมหรือเกมอื่น ๆ ในซีรีส์เดียวกัน เช่น Half-Life: Opposing Force หรือ Counter-Strike: Condition Zero ดังนั้น กฎนี้จึงสะท้อนพฤติกรรมของผู้เล่นที่มีความภักดีต่อแฟรนไชส์และนิยมบริโภคเกมภายในจักรวาลเดียวกัน